# HW 09: When Does Vision Help?
**Computational Sensorimotor Control — Week 9 | **

**Reading:** Crevecoeur, F., Munoz, D. P. & Scott, S. H. (2016). Dynamic multisensory
integration: somatosensory speed trumps visual accuracy during feedback control.
*Journal of Neuroscience*, 36(33), 8598–8611.


In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'serif'})

from smc import Arm, Q_REF, mass_matrix, coriolis, joint_accelerations

arm = Arm()
fk  = arm.forward_kinematics
def ik(p): return arm.inverse_kinematics(p[0], p[1])
Jfn = arm.jacobian
dt  = 0.001

start_hand = np.array(fk(Q_REF))
Kp = 8; k_noise = 0.15

NAVY='#1B2A4A'; TEAL='#2E86AB'; RED='#E74C3C'
GREEN='#27AE60'; GRAY='#7F8C8D'; ORANGE='#E67E22'; PURPLE='#8E44AD'
BLUE='#3498DB'

def add_sdn(tau, k, rng):
    return tau + rng.normal(0, k * np.abs(tau))

def minjerk_reach(tgt, T):
    n = int(T/dt) + 1; t = np.linspace(0, T, n)
    s = 10*(t/T)**3 - 15*(t/T)**4 + 6*(t/T)**5
    return t, start_hand[None,:] + s[:,None] * (tgt - start_hand)[None,:]

def id_torques(t, pos):
    q = np.array([ik(p) for p in pos])
    qd = np.gradient(q, dt, axis=0); qdd = np.gradient(qd, dt, axis=0)
    tau = np.zeros_like(q)
    for i in range(len(q)):
        tau[i] = mass_matrix(q[i]) @ qdd[i] + coriolis(q[i], qd[i])
    return q, qd, qdd, tau


In [ ]:
# ── Simulation functions from Lab 09 ──
def sim_p(q0, tau_ff, pos_des, Kp, delay_s, sigma, rng):
    """Single-sensor P controller (single RNG)."""
    n = len(tau_ff)
    qs = np.zeros((n,2)); qds = np.zeros((n,2)); hs = np.zeros((n,2))
    qs[0] = q0.copy(); hs[0] = fk(q0)
    buf = []; bl = int(delay_s / dt)
    for i in range(n-1):
        buf.append(hs[i].copy()); tc = np.zeros(2)
        if Kp > 0 and len(buf) > bl:
            y = buf[-bl-1] + rng.normal(0, sigma, 2)
            tc = Jfn(qs[i]).T @ (Kp * (pos_des[max(0,i-bl)] - y))
        tt = add_sdn(tau_ff[i] + tc, k_noise, rng)
        qdd_i = joint_accelerations(qs[i], qds[i], tt)
        qds[i+1] = qds[i] + qdd_i*dt
        qs[i+1] = qs[i] + qds[i+1]*dt
        hs[i+1] = fk(qs[i+1])
    return hs

def sim_fused_frontier(q0, tau_ff, pos_des, Kp, sensors, motor_seed):
    """Fused controller with separated RNGs (from Lab 09)."""
    n = len(tau_ff)
    qs = np.zeros((n,2)); qds = np.zeros((n,2)); hs = np.zeros((n,2))
    qs[0] = q0.copy(); hs[0] = fk(q0); buf = []
    rng_motor = np.random.default_rng(motor_seed)
    rng_sensors = [np.random.default_rng(motor_seed*1000+j+1) for j in range(len(sensors))]
    bls = [int(d/dt) for d,_ in sensors]
    sigs = [s for _,s in sensors]
    vel_des_loc = np.gradient(pos_des, dt, axis=0)
    min_bl = min(bls) if sensors else 0
    for i in range(n-1):
        buf.append(hs[i].copy()); tc = np.zeros(2)
        if Kp > 0 and sensors:
            ref_idx = max(0, i - min_bl)
            errs = []; ws = []
            for j, (bl, sig) in enumerate(zip(bls, sigs)):
                if len(buf) > bl:
                    y = buf[-bl-1] + rng_sensors[j].normal(0, sig, 2)
                    if bl > min_bl:
                        proj = y.copy(); src = max(0, i-bl)
                        for k in range(bl - min_bl):
                            proj += vel_des_loc[min(src+k, n-2)] * dt
                        y = proj
                    errs.append(pos_des[ref_idx] - y)
                    ws.append(1.0 / sig**2)
            if errs:
                wt = sum(ws); ef = sum(w*e for w,e in zip(ws,errs)) / wt
                tc = Jfn(qs[i]).T @ (Kp * ef)
        tt = add_sdn(tau_ff[i]+tc, k_noise, rng_motor)
        qdd_i = joint_accelerations(qs[i], qds[i], tt)
        qds[i+1] = qds[i]+qdd_i*dt; qs[i+1] = qs[i]+qds[i+1]*dt; hs[i+1] = fk(qs[i+1])
    return hs


---
## Part A — Analytical

$$\sigma^2_{fused} = \frac{1}{1/\sigma_1^2 + 1/\sigma_2^2}, \quad w_j = \frac{1/\sigma_j^2}{\sum_k 1/\sigma_k^2}$$


In [ ]:
# A1: Compute fused noise and weights for three pairings
pairings = [
    ('Periph + Proprio',  5.0, 12.0),
    ('Central + Proprio', 1.0, 12.0),
    ('Central + Periph',  1.0,  5.0),
]

for name, s1, s2 in pairings:
    sigma_f = ...  # ← YOUR CODE HERE
    w1 = (1/s1**2) / (1/s1**2 + 1/s2**2) * 100
    w2 = (1/s2**2) / (1/s1**2 + 1/s2**2) * 100
    print(f'{name:22s}: σ_fused = {sigma_f:.2f} mm, '
          f'w1 = {w1:.0f}%, w2 = {w2:.0f}%')


**A2:** *(Your answer here — 2–3 sentences)*


---
## Part B — Computational

Run the frontier with two fused pairings:
- Green: proprio + peripheral (Δ=80ms, σ=5mm)
- Blue: proprio + central vision (Δ=150ms, σ=1mm)


In [ ]:
# B1: Run both frontiers
PROPRIO = (0.060, 0.012)
PERIPH  = (0.080, 0.005)
CENTRAL = (0.150, 0.001)  # ← NEW PAIRING

DELAY_PR = 0.060; SIGMA_PR = 0.012
T_vals = [0.25, 0.30, 0.40, 0.50, 0.60, 0.80, 1.0]
N_mc = 150
tgt_r = start_hand + np.array([0.10, 0])

sds = {k: [] for k in ['ol','pr','fu_periph','fu_central']}

for Ti in T_vals:
    t_r, pos_r = minjerk_reach(tgt_r, Ti)
    q_r, _, _, tau_r = id_torques(t_r, pos_r)
    # Open-loop
    eps = np.array([sim_p(q_r[0],tau_r,pos_r,0,0,0,
                          np.random.default_rng(i))[-1] for i in range(N_mc)])
    sds['ol'].append(np.std(np.linalg.norm(eps-tgt_r,axis=1))*100)
    # Proprio only
    eps = np.array([sim_p(q_r[0],tau_r,pos_r,Kp,DELAY_PR,SIGMA_PR,
                          np.random.default_rng(i))[-1] for i in range(N_mc)])
    sds['pr'].append(np.std(np.linalg.norm(eps-tgt_r,axis=1))*100)
    # Fused: proprio + peripheral
    eps = np.array([sim_fused_frontier(q_r[0],tau_r,pos_r,Kp,
                   [PROPRIO,PERIPH],i)[-1] for i in range(N_mc)])
    sds['fu_periph'].append(np.std(np.linalg.norm(eps-tgt_r,axis=1))*100)
    ...  # ← YOUR CODE HERE
    eps = np.array([sim_fused_frontier(q_r[0],tau_r,pos_r,Kp,
    ...  # ← YOUR CODE HERE
    sds['fu_central'].append(np.std(np.linalg.norm(eps-tgt_r,axis=1))*100)
    print(f'  T={Ti*1000:.0f} ms done')


In [ ]:
# B1: Plot
T_ms = [t*1000 for t in T_vals]
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(T_ms, sds['ol'], 'o--', color=RED, lw=2, ms=6, label='Open-loop')
ax.plot(T_ms, sds['pr'], 'D-', color=PURPLE, lw=1.5, ms=5, label=r'Proprio only ($\Delta$=60ms, $\sigma$=12mm)')
ax.plot(T_ms, sds['fu_periph'], 'p-', color=GREEN, lw=2.5, ms=7,
        label='Fused: proprio + peripheral', zorder=5)
ax.plot(T_ms, sds['fu_central'], 's-', color=BLUE, lw=2.5, ms=7,
        label='Fused: proprio + central', zorder=5)
ax.set_xlabel('Movement time (ms)'); ax.set_ylabel('Endpoint SD (cm)')
ax.set_title('HW9 Part B: Two fused frontiers — peripheral vs central vision', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(200, 1050)
plt.tight_layout(); plt.show()


In [ ]:
# B2: Report numerical values at T=400ms and T=1000ms
for Ti_target in [400, 1000]:
    idx = T_vals.index(Ti_target/1000)
    print(f'T = {Ti_target} ms:')
    print(f'  Proprio only:         {sds["pr"][idx]:.4f} cm')
    print(f'  Fused (periph):       {sds["fu_periph"][idx]:.4f} cm')
    print(f'  Fused (central):      {sds["fu_central"][idx]:.4f} cm')
    print()


**B3:** *(Your answer here — 2–3 sentences)*


---
## Part C — Reading


In [ ]:
# C1 + C2: Percentage improvement from adding vision
for Ti_target in [400, 1000]:
    idx = T_vals.index(Ti_target/1000)
    sd_pr = sds['pr'][idx]
    sd_periph = sds['fu_periph'][idx]
    sd_central = sds['fu_central'][idx]
    pct_periph = (sd_pr - sd_periph) / sd_pr * 100
    pct_central = (sd_pr - sd_central) / sd_pr * 100
    print(f'T = {Ti_target} ms:')
    print(f'  Adding peripheral: {pct_periph:+.1f}% (SD: {sd_pr:.4f} -> {sd_periph:.4f})')
    print(f'  Adding central:    {pct_central:+.1f}% (SD: {sd_pr:.4f} -> {sd_central:.4f})')
    print()


**C1:** *(Your answer here)*

**C2:** *(Your answer here)*

**C3:** *(Your answer here — 2–3 sentences on gaze strategy)*
